# Bias-Variance Tradeoff

**Goal:** Empirically simulate the bias-variance decomposition: generate noisy data from a known function, fit polynomial models of increasing degree across many resampled datasets, estimate bias², variance, and irreducible noise, and plot the U-shape of total error vs complexity.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Setup: True Function, Noise, and Simulation Parameters

We use a sinusoidal true function `f(x) = sin(2πx)` on [0, 1] with additive Gaussian noise σ_noise. We fit polynomial regression models of degree d = 1, …, 10 on `B` independently drawn (Monte Carlo) training sets, then estimate at a fixed test grid.

In [2]:
# All computation on CPU for polynomial fitting
# (torch.linalg.lstsq is well-supported on CPU)
cpu = torch.device("cpu")

torch.manual_seed(42)

SIGMA_NOISE = 0.3   # irreducible noise level
N_TRAIN = 30        # samples per training set
N_BOOTSTRAPS = 500  # number of independently drawn (Monte Carlo) training sets
MAX_DEGREE = 10     # max polynomial degree to try
N_TEST = 200        # test points for evaluation


def true_fn(x: torch.Tensor) -> torch.Tensor:
    """True noiseless function: sin(2*pi*x)."""
    return torch.sin(2 * torch.pi * x)


# Fixed test grid
x_test = torch.linspace(0.0, 1.0, N_TEST, device=cpu)
y_test_true = true_fn(x_test)  # f(x) without noise

# Plot true function
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x_test.numpy(), y_test_true.numpy(), "k-", lw=2, label="f(x) = sin(2πx)")
# Show one sample training set
x_sample = torch.rand(N_TRAIN, device=cpu)
y_sample = true_fn(x_sample) + torch.randn(N_TRAIN, device=cpu) * SIGMA_NOISE
ax.scatter(x_sample.numpy(), y_sample.numpy(), s=20, alpha=0.7, label="training sample")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("True function and one noisy training set")
ax.legend()
plt.tight_layout()
plt.show()

/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_14805/1901109294.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Polynomial Feature Matrix and Least-Squares Fit

We map x → [1, x, x², …, xᵈ] and solve the normal equations with `torch.linalg.lstsq`.

In [3]:
def poly_features(x: torch.Tensor, degree: int) -> torch.Tensor:
    """Build polynomial design matrix: shape (n, degree+1)."""
    return torch.stack([x**d for d in range(degree + 1)], dim=1)


def fit_poly(x_tr: torch.Tensor, y_tr: torch.Tensor, degree: int) -> torch.Tensor:
    """Fit polynomial of given degree; return coefficient vector."""
    X = poly_features(x_tr, degree)
    # lstsq returns a namedtuple; solution is .solution
    result = torch.linalg.lstsq(X, y_tr.unsqueeze(1))
    return result.solution.squeeze(1)


def predict_poly(x: torch.Tensor, coef: torch.Tensor) -> torch.Tensor:
    """Predict using polynomial coefficients."""
    degree = coef.numel() - 1
    X = poly_features(x, degree)
    return X @ coef


# Quick sanity check: degree-1 fit
coef_d1 = fit_poly(x_sample, y_sample, degree=1)
print(f"Degree-1 coefficients: {coef_d1.cpu().tolist()}")

Degree-1 coefficients: [1.314852237701416, -2.3742499351501465]


## Bias-Variance Simulation

For each polynomial degree d and each of B resampled training sets, we:
1. Draw N_TRAIN points with noise.
2. Fit degree-d polynomial.
3. Predict at x_test.

Then empirically estimate:
```
Bias²(x)   = (mean prediction - f(x))²
Variance(x) = mean (prediction - mean prediction)²
Noise       = σ_noise²  (irreducible)
Total MSE   = Bias² + Variance + Noise  (theoretically)
```

In [4]:
degrees = list(range(1, MAX_DEGREE + 1))

bias2_list: list[float] = []
var_list: list[float] = []
total_mse_list: list[float] = []
noise_level = SIGMA_NOISE**2  # irreducible noise sigma^2

torch.manual_seed(42)

for degree in degrees:
    # Collect predictions across B independently drawn (Monte Carlo) training sets
    preds = []  # each entry shape (N_TEST,)
    noisy_targets = []  # independently drawn noisy test targets
    for _ in range(N_BOOTSTRAPS):
        x_tr = torch.rand(N_TRAIN, device=cpu)
        y_tr = true_fn(x_tr) + torch.randn(N_TRAIN, device=cpu) * SIGMA_NOISE
        coef = fit_poly(x_tr, y_tr, degree)
        preds.append(predict_poly(x_test, coef))
        # Draw fresh independent noise for the test labels this iteration
        noisy_targets.append(y_test_true + torch.randn(N_TEST, device=cpu) * SIGMA_NOISE)

    # Stack: shape (N_BOOTSTRAPS, N_TEST)  # noqa: N_BOOTSTRAPS kept for compatibility
    preds_t = torch.stack(preds, dim=0)
    targets_t = torch.stack(noisy_targets, dim=0)  # noisy test labels

    mean_pred = preds_t.mean(dim=0)         # mean over Monte Carlo training sets
    bias2 = ((mean_pred - y_test_true) ** 2).mean().item()
    variance = preds_t.var(dim=0, unbiased=False).mean().item()
    # Total MSE computed against noisy targets so it equals bias²+var+noise
    total_mse = ((preds_t - targets_t) ** 2).mean().item()

    bias2_list.append(bias2)
    var_list.append(variance)
    total_mse_list.append(total_mse)

print("Simulation complete.")
print(f"{'Degree':>6} {'Bias²':>10} {'Variance':>10} {'Total MSE':>10} {'B²+V+σ²':>12}")
for d, b2, v, mse in zip(degrees, bias2_list, var_list, total_mse_list):
    approx = b2 + v + noise_level
    print(f"{d:>6}   {b2:>8.5f}   {v:>8.5f}   {mse:>8.5f}   {approx:>10.5f}")

Simulation complete.
Degree      Bias²   Variance  Total MSE      B²+V+σ²
     1    0.19988    0.02120    0.31145      0.31108
     2    0.20181    0.04837    0.33948      0.34017
     3    0.00486    0.01850    0.11368      0.11336
     4    0.00501    0.02613    0.12109      0.12114
     5    0.00009    0.06270    0.15299      0.15279
     6    0.00045    0.21282    0.30326      0.30327
     7    0.00033    0.17104    0.25991      0.26137
     8    0.00022    0.31636    0.40710      0.40658
     9    0.00056    0.33891    0.43060      0.42947
    10    0.00018    1.38867    1.47975      1.47885


In [5]:
# Assert Bias² + Variance + sigma² ≈ Total MSE for each degree
# total_mse is measured against noisy test targets, so:
#   E[(y_noisy - f_hat)²] = E[(f(x)+ε - f_hat)²] = bias² + var + σ²
for d, b2, v, mse in zip(degrees, bias2_list, var_list, total_mse_list):
    decomp = b2 + v + noise_level
    assert abs(decomp - mse) < 0.005, (
        f"Degree {d}: Bias²+Var+noise={decomp:.5f} vs total MSE={mse:.5f}"
    )
print("Bias² + Variance + σ² ≈ Total MSE for all degrees ✓")


Bias² + Variance + σ² ≈ Total MSE for all degrees ✓


In [6]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(degrees, bias2_list, "b-o", ms=5, label="Bias²")
ax.plot(degrees, var_list, "g-s", ms=5, label="Variance")
ax.plot(degrees, total_mse_list, "r-^", ms=5, label="Total MSE")
ax.axhline(noise_level, color="gray", ls="--", label=f"Irreducible noise σ²={noise_level:.3f}")
ax.set_xlabel("Polynomial Degree (model complexity)")
ax.set_ylabel("Error")
ax.set_title("Bias-Variance Tradeoff: Total Error vs Model Complexity")
ax.legend()
plt.tight_layout()
plt.show()

best_degree = degrees[total_mse_list.index(min(total_mse_list))]
print(f"Optimal polynomial degree: {best_degree} (lowest total MSE = {min(total_mse_list):.5f})")

Optimal polynomial degree: 3 (lowest total MSE = 0.11368)


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_14805/2625609490.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Visual: Underfit vs Optimal vs Overfit

Plot predictions from a single training set for a low-degree (underfit), optimal-degree, and high-degree (overfit) model.

In [7]:
torch.manual_seed(42)
x_vis = torch.rand(N_TRAIN, device=cpu)
y_vis = true_fn(x_vis) + torch.randn(N_TRAIN, device=cpu) * SIGMA_NOISE

example_degrees = [1, best_degree, 9]
labels = ["Underfit (d=1)", f"Near-optimal (d={best_degree})", "Overfit (d=9)"]
colors = ["blue", "green", "red"]

x_plot = torch.linspace(0.0, 1.0, 300, device=cpu)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, d, label, color in zip(axes, example_degrees, labels, colors):
    coef = fit_poly(x_vis, y_vis, d)
    y_pred = predict_poly(x_plot, coef)
    ax.scatter(x_vis.numpy(), y_vis.numpy(), s=20, alpha=0.6, color="gray", zorder=3)
    ax.plot(x_plot.numpy(), true_fn(x_plot).numpy(), "k--", lw=1.5, label="true f(x)")
    ax.plot(x_plot.numpy(), y_pred.numpy(), color=color, lw=2, label=label)
    ax.set_title(label)
    ax.set_xlabel("x")
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=8)

axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_14805/3274965542.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways

- **Decomposition:** Total MSE = Bias² + Variance + σ² (irreducible noise). We verified this empirically across B=500 resampled training sets.
- **U-shape:** Increasing model complexity decreases bias but increases variance. The optimal degree minimises total error.
- **Underfit vs overfit:** A linear model on sin(2πx) has high bias everywhere. A degree-9 polynomial fits training data closely but oscillates wildly.
- **Irreducible noise (σ²):** Cannot be removed by any model. It is a lower bound on MSE.
- **AI engineering connection:** This is why you need a validation set — training loss only reveals bias; variance requires comparing models across different data splits.